# Cuaderno U0-00. Puesta a punto del entorno de trabajo

**Modelacion y Simulacion Computacional** · Maestria en Ingenieria · Universidad de Sucre

Unidad 1, Fundamentos de modelacion en ingenieria · Cuaderno comun previo a las cuatro unidades

Docente Daniel David Otero Meza · Periodo 2026-2

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/U0_00_puesta_a_punto.ipynb)

*El docente reemplaza `msc-unisucre/msc2026-material` por la direccion real del repositorio del
curso antes de publicar el cuaderno.*

Este cuaderno se ejecuta una sola vez, antes de la Unidad 1, y deja el entorno en las condiciones que exigen los demas cuadernos del curso. Comprueba las bibliotecas, fija la semilla, define la paleta del libro, explica como se organiza un proyecto reproducible y termina con una autoevaluacion que dice si el entorno esta listo. Funciona igual en Google Colab y en JupyterLab local.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante estara en capacidad de hacer lo siguiente.

1. Verificar que NumPy, SciPy, pandas, Matplotlib y SymPy estan disponibles en una version compatible con los cuadernos del curso.
2. Fijar la semilla de la asignatura y demostrar que dos corridas independientes producen exactamente los mismos numeros.
3. Usar la paleta del libro en toda figura que se entregue.
4. Localizar los archivos de la carpeta datos sin escribir rutas absolutas y regenerarlos con la semilla cuando no esten.
5. Aplicar el flujo minimo de Git que la asignatura exige en cada entrega.

## Puesta a punto

La primera celda detecta el entorno e instala unicamente lo que falte. La segunda fija la semilla del curso y la paleta del libro. La tercera define las funciones de verificacion que se usan mas abajo. Ejecutelas en orden antes de continuar.

In [ ]:
# Puesta a punto del entorno. Detecta Colab e instala solo lo que falte.
import importlib
import importlib.util
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes):
    """Instala los paquetes ausentes sin reinstalar los que ya estan."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)
    return faltantes


AUSENTES = asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                     "matplotlib": "matplotlib", "sympy": "sympy"})

print("Entorno de ejecucion:", "Google Colab" if EN_COLAB else "JupyterLab local")
print("Paquetes instalados en esta sesion:", AUSENTES or "ninguno, ya estaban")

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

# Semilla unica de la asignatura. Ningun resultado depende de una corrida.
SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

# Paleta del libro. Los cuadernos usan los mismos colores que las figuras.
PALETA = {
    "azul": "#1F4E79",
    "rojo": "#B3251E",
    "verde": "#2E7D32",
    "naranja": "#E07B00",
    "gris": "#5A5A5A",
    "morado": "#6A3D9A",
}

plt.rcParams.update({
    "figure.figsize": (8.6, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
    "font.size": 10.0,
    "legend.frameon": True,
    "legend.framealpha": 0.92,
})

print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · pandas {pd.__version__}")
print(f"SymPy {sp.__version__} · semilla del curso {SEMILLA}")

In [ ]:
# Bandera de revision de los ejercicios guiados.
# Mientras valga False el cuaderno se ejecuta completo aunque falten celdas.
# Pongala en True cuando haya completado las celdas marcadas para completar.
REVISAR = False
print("REVISAR =", REVISAR)

In [ ]:
def verificar_libro(nombre, obtenido, publicado, tolerancia=1e-3, unidad=""):
    """Contrasta un resultado calculado con la cifra que publica el libro."""
    valor = float(obtenido)
    escala = abs(publicado) if publicado else 1.0
    error = abs(valor - publicado) / escala
    print(f"{nombre:<46s} calculado {valor:>12.6g} {unidad:<10s}"
          f" libro {publicado:>12.6g}  error rel. {error:.1e}")
    assert error <= tolerancia, f"{nombre} se aparta de la cifra publicada"
    return valor


def comprobar(nombre, obtenido, referencia, tolerancia=1e-3, unidad=""):
    """Revisa una celda de ejercicio contra su valor de referencia.

    Con REVISAR en False solo informa que el ejercicio sigue pendiente, de modo
    que el cuaderno nunca se detiene por una celda sin completar.
    """
    if not REVISAR:
        print(f"[pendiente]  {nombre}")
        return False
    valor = float(obtenido)
    escala = abs(referencia) if referencia else 1.0
    error = abs(valor - referencia) / escala
    marca = "correcto " if error <= tolerancia else "revisar  "
    print(f"[{marca}]  {nombre} = {valor:.6g} {unidad}"
          f"  referencia {referencia:.6g}  error rel. {error:.2e}")
    assert error <= tolerancia, f"{nombre} no coincide con la referencia"
    return True


def ruta_datos(nombre):
    """Ubica un archivo de la carpeta datos sin usar rutas absolutas.

    Funciona igual en Colab, donde el cuaderno suele abrirse en el directorio
    de trabajo, y en una copia local del repositorio, donde el cuaderno vive
    dentro de Unidad1 o de soluciones.
    """
    candidatas = (Path("datos"),
                  Path("..") / "datos",
                  Path("..") / ".." / "datos",
                  Path("03_cuadernos") / "datos")
    for base in candidatas:
        if (base / nombre).exists():
            return base / nombre
    for base in candidatas:        # la carpeta existe pero el archivo aun no
        if base.is_dir():
            return base / nombre
    return Path("datos") / nombre  # entorno nuevo, como una sesion de Colab


def cargar_o_generar(nombre, generador):
    """Lee el archivo de datos y, si no esta, lo reconstruye con la semilla."""
    ruta = ruta_datos(nombre)
    if ruta.exists():
        print(f"Datos leidos de {ruta}")
        return pd.read_csv(ruta)
    tabla = generador()
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(ruta, index=False)
    print(f"Datos regenerados con la semilla {SEMILLA} y guardados en {ruta}")
    return tabla


def integrar_trapecio(valores, muestras):
    """Regla del trapecio compatible con NumPy 1 y con NumPy 2."""
    regla = getattr(np, "trapezoid", None) or np.trapz
    return float(regla(valores, muestras))


print("Funciones auxiliares disponibles.")

## 1. Versiones de las bibliotecas del curso

La asignatura usa cinco bibliotecas y nada mas. NumPy para el calculo numerico,
SciPy para raices, integracion y funciones especiales, pandas para tablas,
Matplotlib para figuras y SymPy para algebra simbolica. La celda siguiente
compara la version instalada con la version minima que los cuadernos suponen y
deja el resultado en una tabla.

Una version por debajo de la minima no detiene el cuaderno, pero conviene
actualizarla antes de empezar la Unidad 1.

In [ ]:
BITACORA = []


def registrar(concepto, correcto, detalle=""):
    """Anota el resultado de una comprobacion para la autoevaluacion final."""
    BITACORA.append({"comprobacion": concepto,
                     "estado": "correcto" if correcto else "revisar",
                     "detalle": detalle})
    return correcto


def version_tupla(texto):
    """Convierte una cadena de version en una tupla comparable."""
    partes = []
    for trozo in str(texto).split(".")[:3]:
        digitos = "".join(c for c in trozo if c.isdigit())
        partes.append(int(digitos) if digitos else 0)
    return tuple(partes + [0] * (3 - len(partes)))


MINIMAS = {
    "NumPy": (np.__version__, "1.24.0"),
    "SciPy": (scipy.__version__, "1.10.0"),
    "pandas": (pd.__version__, "1.5.0"),
    "Matplotlib": (plt.matplotlib.__version__, "3.6.0"),
    "SymPy": (sp.__version__, "1.11.0"),
}

filas = []
for biblioteca, (instalada, minima) in MINIMAS.items():
    suficiente = version_tupla(instalada) >= version_tupla(minima)
    registrar(f"version de {biblioteca}", suficiente, f"{instalada} frente a {minima}")
    filas.append({"biblioteca": biblioteca, "instalada": instalada,
                  "minima requerida": minima,
                  "estado": "correcto" if suficiente else "actualizar"})

versiones = pd.DataFrame(filas)
print(f"Python {sys.version.split()[0]}")
versiones

## 2. La semilla del curso

Todo resultado aleatorio de la asignatura se genera con la semilla
`SEMILLA = 20262`. La regla es estricta porque un informe cuyo numero cambia en
cada corrida no es reproducible y no se puede revisar.

Conviene distinguir dos cosas que se confunden con frecuencia. Un generador
creado con la semilla siempre entrega la misma secuencia, y un generador ya
usado avanza su estado interno, de modo que la segunda llamada no repite la
primera. La celda siguiente lo demuestra con ambas situaciones.

El curso usa `np.random.default_rng`, que es la interfaz moderna de NumPy, y no
`np.random.seed`, que actua sobre un estado global compartido y hace muy dificil
rastrear de donde sale cada numero.

In [ ]:
primero = np.random.default_rng(SEMILLA).normal(size=5)
segundo = np.random.default_rng(SEMILLA).normal(size=5)
identicos = bool(np.allclose(primero, segundo))

generador = np.random.default_rng(SEMILLA)
tanda_a = generador.normal(size=5)
tanda_b = generador.normal(size=5)
avanza = not bool(np.allclose(tanda_a, tanda_b))

print("Dos generadores con la misma semilla")
print("  ", np.round(primero, 6))
print("  ", np.round(segundo, 6))
print("Coinciden:", identicos)
print()
print("Un mismo generador llamado dos veces")
print("  ", np.round(tanda_a, 6))
print("  ", np.round(tanda_b, 6))
print("Avanza el estado interno:", avanza)

registrar("reproducibilidad de la semilla", identicos and avanza,
          "misma semilla, misma secuencia")
assert identicos and avanza

## 3. La paleta del curso

Las figuras del libro y de los cuadernos usan seis colores fijos. Emplear
siempre la misma paleta no es un capricho estetico, sino una forma de que el
lector asocie un color con un significado a lo largo de todo el material, por
ejemplo el azul para el modelo de referencia y el rojo para la alternativa que
se cuestiona.

El estilo tipografico del libro no se reproduce aqui, porque exige LaTeX, y en
su lugar basta con rotulos claros y unidades explicitas en los dos ejes.

In [ ]:
figura, eje = plt.subplots(figsize=(8.6, 2.3))
for i, (nombre, codigo) in enumerate(PALETA.items()):
    eje.bar(i, 1.0, color=codigo, width=0.82)
    eje.text(i, 0.5, f"{nombre}\n{codigo}", ha="center", va="center",
             color="white", fontsize=9.5)
eje.set_xticks([])
eje.set_yticks([])
eje.grid(False)
eje.set_title("Paleta de la asignatura")
plt.show()

correcta = list(PALETA.values()) == ["#1F4E79", "#B3251E", "#2E7D32",
                                     "#E07B00", "#5A5A5A", "#6A3D9A"]
registrar("paleta del curso", correcta, ", ".join(PALETA.values()))
print("Paleta cargada:", correcta)

## 4. Estructura del proyecto reproducible

Un proyecto reproducible es aquel en el que un tercero, con el repositorio y
nada mas, obtiene los mismos numeros que estan en el informe. Eso exige separar
lo que se escribe a mano de lo que se genera, y no guardar nunca una ruta del
computador de quien lo escribio.

La estructura minima que la asignatura pide para el mini proyecto es la que
imprime la celda siguiente. Los datos crudos no se modifican jamas, los datos
procesados se pueden borrar y volver a generar, y las figuras y tablas del
informe salen siempre de un cuaderno o de un script, nunca de una edicion
manual.

La funcion `ruta_datos` resuelve el problema de las rutas. Prueba las
ubicaciones habituales en orden y devuelve la primera que exista, de modo que el
mismo cuaderno funciona abierto en Colab, donde el directorio de trabajo es la
raiz de la sesion, y abierto en una copia local, donde el cuaderno vive dentro de
`Unidad1` o de `soluciones`. Si el archivo no aparece por ningun lado,
`cargar_o_generar` lo reconstruye con la semilla del curso y lo guarda.

In [ ]:
ESTRUCTURA = """
mi_proyecto/
  README.md              que es, como se ejecuta y quien responde por el
  requirements.txt       versiones exactas de las bibliotecas
  .gitignore             lo que no entra al repositorio
  datos/
    crudos/              tal como se recibieron, solo lectura
    procesados/          derivados, se pueden borrar y regenerar
  cuadernos/             exploracion y desarrollo
  src/                   funciones reutilizables, probadas
  figuras/               salidas del codigo, nunca editadas a mano
  informe/               documento final reproducible
"""
print(ESTRUCTURA)


def generar_series_demostracion():
    """Serie horaria sintetica de un dia, generada con la semilla del curso."""
    generador = np.random.default_rng(SEMILLA)
    hora = np.arange(24)
    temperatura = (27.0 + 5.0 * np.sin(np.pi * (hora - 7) / 12)
                   + generador.normal(0.0, 0.35, 24))
    irradiancia = np.clip(950.0 * np.sin(np.pi * (hora - 6) / 12), 0.0, None)
    jornada = np.where(hora < 6, "00-06",
                       np.where(hora < 12, "06-12",
                                np.where(hora < 18, "12-18", "18-24")))
    return pd.DataFrame({"hora": hora,
                         "jornada": jornada,
                         "temperatura_C": np.round(temperatura, 3),
                         "irradiancia_W_m2": np.round(irradiancia, 1)})


series = cargar_o_generar("U0_series_demostracion.csv", generar_series_demostracion)
print(f"\nLa tabla tiene {len(series)} filas y {series.shape[1]} columnas.")
registrar("carga de datos sin rutas absolutas", len(series) == 24,
          str(ruta_datos("U0_series_demostracion.csv")))
series.head()

## 5. Flujo minimo de Git

La asignatura evalua el repositorio, no solo el resultado. El flujo minimo que
se exige en cada entrega tiene siete pasos y cabe en una sesion de cinco
minutos.

1. `git status` para ver que cambio desde el ultimo registro.
2. `git add cuadernos/U1_01.ipynb` para escoger que entra, archivo por archivo.
3. `git commit -m "U1-01, reproduce el ejemplo de la estacion de bombeo"` con un
   mensaje que diga que se hizo y por que, en presente y sin rodeos.
4. `git log --oneline -5` para confirmar que el historial se lee.
5. `git switch -c ejercicio-canal` cuando se va a probar algo que puede fallar.
6. `git switch main` y `git merge ejercicio-canal` cuando la prueba salio bien.
7. `git push` para que el docente vea el trabajo.

Tres reglas que evitan casi todos los problemas del curso. La primera es no
subir archivos generados, para lo cual el `.gitignore` debe excluir
`__pycache__/`, `.ipynb_checkpoints/`, `datos/procesados/` y las figuras. La
segunda es limpiar las salidas de los cuadernos antes de registrarlos, porque
las salidas convierten cualquier revision en un conflicto ilegible. La tercera
es que un registro por sesion de trabajo es demasiado poco y un registro por
linea es demasiado, de modo que la unidad razonable es un cambio que se pueda
describir en una frase.

In [ ]:
COMANDOS = [
    ("git status", "que cambio desde el ultimo registro"),
    ("git add ruta/al/archivo", "escoger que entra al registro"),
    ("git commit -m \"mensaje en presente\"", "registrar el cambio"),
    ("git log --oneline -5", "revisar el historial reciente"),
    ("git switch -c nombre-de-la-rama", "abrir una rama para una prueba"),
    ("git switch main", "volver a la rama principal"),
    ("git merge nombre-de-la-rama", "incorporar la prueba que funciono"),
    ("git push", "publicar el trabajo"),
]
flujo = pd.DataFrame(COMANDOS, columns=["comando", "para que sirve"])

try:
    salida = subprocess.run(["git", "--version"], capture_output=True,
                            text=True, timeout=20)
    hay_git = salida.returncode == 0
    detalle = salida.stdout.strip()
except (FileNotFoundError, subprocess.TimeoutExpired):
    hay_git = False
    detalle = "git no esta instalado en este entorno"

print(detalle if hay_git else "Aviso, " + detalle)
if not hay_git:
    print("El cuaderno sigue funcionando, pero la entrega exige un repositorio.")
registrar("git disponible", hay_git, detalle)
flujo

## 6. Prueba de las cinco bibliotecas

Comprobar que una biblioteca importa no basta, porque una instalacion
incompleta importa y falla al calcular. La celda siguiente ejecuta una operacion
representativa de cada una y verifica su resultado contra un valor conocido de
forma independiente, que es exactamente la disciplina que el resto del curso
aplica a los modelos.

In [ ]:
from scipy.integrate import quad
from scipy.optimize import brentq

# NumPy, resolver un sistema lineal y comprobar el residuo.
matriz = np.array([[4.0, -2.0, 1.0], [-2.0, 5.0, 3.0], [1.0, 3.0, 7.0]])
lado = np.array([11.0, 3.0, 26.0])
solucion = np.linalg.solve(matriz, lado)
residuo = float(np.max(np.abs(matriz @ solucion - lado)))
registrar("NumPy, sistema lineal", residuo < 1e-10, f"residuo {residuo:.2e}")

# SciPy, una raiz y una integral de valor conocido.
raiz = brentq(lambda x: x**2 - 2.0, 0.0, 2.0, xtol=1e-14)
integral, _ = quad(np.sin, 0.0, np.pi)
registrar("SciPy, raiz e integral",
          abs(raiz - np.sqrt(2)) < 1e-12 and abs(integral - 2.0) < 1e-10,
          f"raiz {raiz:.12f}, integral {integral:.12f}")

# pandas, una agrupacion cuyo resultado se conoce.
medias = series.groupby("jornada")["temperatura_C"].mean()
registrar("pandas, agrupacion", len(medias) == 4,
          ", ".join(f"{k} {v:.2f}" for k, v in medias.items()))

# SymPy, una derivada simbolica y su valor exacto.
x = sp.Symbol("x", positive=True)
derivada = sp.diff(sp.log(x) * sp.sqrt(x), x)
valor = sp.simplify(derivada.subs(x, 1))
registrar("SymPy, derivada simbolica", valor == sp.Rational(1, 1),
          f"d/dx de log(x)*sqrt(x) en x=1 vale {valor}")

# Matplotlib, una figura que se dibuja de verdad.
malla = np.linspace(0.0, 2 * np.pi, 200)
figura, eje = plt.subplots(figsize=(8.6, 3.0))
eje.plot(malla, np.sin(malla), color=PALETA["azul"], label="seno")
eje.plot(malla, np.cos(malla), color=PALETA["rojo"], ls="--", label="coseno")
eje.set_xlabel("Angulo (rad)")
eje.set_ylabel("Valor (adimensional)")
eje.legend(loc="upper right")
plt.show()
registrar("Matplotlib, figura", len(eje.lines) == 2, "dos curvas dibujadas")

print(f"raiz de 2 {raiz:.12f} · integral del seno {integral:.12f} · "
      f"residuo lineal {residuo:.2e}")
print(f"temperatura media por jornada\n{medias.round(3)}")

## 7. Ejercicios guiados

Las celdas siguientes estan incompletas a proposito. Cada una lleva la marca
`# COMPLETE:` con la descripcion de lo que falta, un valor de partida
deliberadamente incorrecto y, justo despues, una celda de verificacion.

Complete las cinco celdas, vuelva a la celda de la bandera, ponga
`REVISAR = True` y ejecute el cuaderno de nuevo desde el principio. Mientras la
bandera valga `False` las verificaciones solo informan que el ejercicio sigue
pendiente y el cuaderno se ejecuta completo sin detenerse.

### Ejercicio 1. La semilla en accion

In [ ]:
# COMPLETE: cree un generador propio con la semilla del curso, extraiga una
# muestra de 1000 valores de una distribucion normal de media 12.5 y desviacion
# estandar 2.0, y deje la media muestral en media_muestral y la desviacion
# muestral, calculada con ddof=1, en desviacion_muestral.
media_muestral = 0.0            # valor de partida deliberadamente incorrecto
desviacion_muestral = 0.0       # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("media muestral", media_muestral, 12.448163, 1e-5)
dos = comprobar("desviacion muestral", desviacion_muestral, 2.050471, 1e-5)
registrar("ejercicio 1, semilla", bool(uno and dos))

### Ejercicio 2. Datos sin rutas absolutas

In [ ]:
# COMPLETE: use cargar_o_generar con el archivo U0_series_demostracion.csv y el
# generador generar_series_demostracion, y calcule la temperatura media de todo
# el dia en temperatura_media y la temperatura media de la jornada 12-18 en
# temperatura_tarde. Las dos van en grados Celsius.
temperatura_media = 0.0         # valor de partida deliberadamente incorrecto
temperatura_tarde = 0.0         # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("temperatura media del dia", temperatura_media, 27.007958, 1e-5, "C")
dos = comprobar("temperatura media de 12-18", temperatura_tarde, 31.152833, 1e-5, "C")
registrar("ejercicio 2, datos", bool(uno and dos))

### Ejercicio 3. Una raiz con SciPy

La ecuacion caracteristica de la conduccion transitoria en una placa plana es
`z*tan(z) = Bi`, y su primera raiz positiva se necesita en el Ejemplo 1.4 del
libro. Con un numero de Biot de 0.0923077 esa raiz vale poco menos de 0.3.
Resuelvala con `brentq`, cuidando que el intervalo de busqueda no incluya la
asintota en pi medios.

In [ ]:
# COMPLETE: halle con brentq la primera raiz positiva de z*tan(z) - Bi para
# Bi = 0.0923077, buscando dentro del intervalo abierto entre 0 y pi/2, y deje
# el resultado en zeta_uno.
BIOT = 0.0923077
zeta_uno = 0.0                  # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("primera raiz de z tan z = Bi", zeta_uno, 0.299226, 1e-5)
registrar("ejercicio 3, SciPy", bool(uno))

### Ejercicio 4. Una derivada con SymPy

In [ ]:
# COMPLETE: derive de forma simbolica la funcion exp(-x**2/2) respecto de x,
# evalue la derivada en x = 1 y deje el valor numerico en derivada_en_uno.
derivada_en_uno = 0.0           # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("derivada de exp(-x^2/2) en x = 1", derivada_en_uno, -0.60653066, 1e-6)
registrar("ejercicio 4, SymPy", bool(uno))

### Ejercicio 5. Una figura con la paleta del curso

In [ ]:
# COMPLETE: dibuje en unos mismos ejes la funcion seno con el color azul de la
# paleta y la funcion coseno con el color rojo, sobre el intervalo de 0 a 2 pi,
# con rotulo en los dos ejes, unidades y leyenda. Deje los ejes en el nombre
# eje_ejercicio y trace primero el seno.
figura_ejercicio, eje_ejercicio = plt.subplots(figsize=(8.6, 3.0))
# aqui faltan las dos curvas, los rotulos y la leyenda
plt.show()

In [ ]:
from matplotlib.colors import to_hex

trazos = list(eje_ejercicio.lines)
colores = [to_hex(t.get_color()).upper() for t in trazos]
aciertos = int(len(trazos) == 2)
aciertos += int(len(colores) > 0 and colores[0] == PALETA["azul"].upper())
aciertos += int(len(colores) > 1 and colores[1] == PALETA["rojo"].upper())
aciertos += int(bool(eje_ejercicio.get_xlabel()) and bool(eje_ejercicio.get_ylabel()))
print("colores usados:", colores or "ninguno")
uno = comprobar("aciertos de la figura, sobre 4", aciertos, 4, 1e-9)
registrar("ejercicio 5, Matplotlib", bool(uno))

## 8. Autoevaluacion del entorno

La celda final reune todas las comprobaciones de este cuaderno y emite un
dictamen. El entorno esta listo cuando las comprobaciones de bibliotecas,
semilla, paleta, datos y figuras estan en verde. Las comprobaciones de los cinco
ejercicios quedan en estado pendiente hasta que se completen las celdas y se
ponga `REVISAR = True`, lo cual no impide empezar la Unidad 1 pero si conviene
resolver antes de la primera sesion.

In [ ]:
resumen = pd.DataFrame(BITACORA)
esenciales = resumen[~resumen["comprobacion"].str.startswith("ejercicio")]
ejercicios = resumen[resumen["comprobacion"].str.startswith("ejercicio")]

entorno_listo = bool((esenciales["estado"] == "correcto").all())
ejercicios_listos = bool(len(ejercicios) > 0
                         and (ejercicios["estado"] == "correcto").all())

print("=" * 72)
print("AUTOEVALUACION DEL ENTORNO")
print("=" * 72)
for _, fila in resumen.iterrows():
    marca = "[ok]     " if fila["estado"] == "correcto" else "[revisar]"
    print(f"{marca} {fila['comprobacion']:<42s} {fila['detalle']}")
print("-" * 72)

if entorno_listo and ejercicios_listos:
    print("Entorno listo y ejercicios resueltos. Puede pasar al cuaderno U1-01.")
elif entorno_listo:
    print("Entorno listo. Faltan ejercicios por completar, que no bloquean la "
          "Unidad 1 pero se revisan en la primera sesion.")
else:
    print("Entorno incompleto. Revise las filas marcadas antes de continuar y, "
          "si el problema es una version, actualicela con pip install -U.")
print("=" * 72)

resumen

## Cierre

### Lista de comprobacion

Marque cada punto solo si puede hacerlo sin mirar el cuaderno.

- Instalar lo que falte con el bloque `asegurar` sin reinstalar lo que ya esta.
- Explicar por que dos generadores con la misma semilla coinciden y un mismo generador llamado dos veces no.
- Dibujar una figura con la paleta del curso y con unidades en los dos ejes.
- Localizar un archivo de la carpeta datos sin escribir una sola ruta absoluta y regenerarlo cuando no exista.
- Registrar un cambio en Git con un mensaje que un tercero entienda.

### Que revisar si algo no salio

- Si una version quedo por debajo de la minima, actualicela con `pip install -U nombre` y vuelva a ejecutar el cuaderno completo.
- Si la lectura de datos falla, confirme desde que carpeta abrio el cuaderno, porque `ruta_datos` prueba `datos`, `../datos` y `../../datos` en ese orden.
- Si Git no aparece, en Colab se instala con el gestor de paquetes del sistema desde una celda de shell, y en local se descarga de git-scm.com.
- Si una figura sale sin colores del curso, revise que paso el argumento `color=PALETA[...]` y no un nombre de color de Matplotlib.

### Declaracion del uso de asistentes de programacion

Si empleo un asistente basado en modelos de lenguaje para resolver alguna celda, declarelo en la entrega, indique en cual y describa que prueba aplico para convencerse de que el codigo es correcto. La regla de la asignatura es que el estudiante responde por el resultado que firma, con independencia de quien escriba las lineas.